In [1]:
!pip install groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 6.4 MB/s eta 0:00:00


In [2]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")


Enter your Groq API Key: ··········


In [3]:
BASE_MODEL = "llama-3.1-8b-instant"

MODEL_CONFIG = {
    "technical": {
        "model": BASE_MODEL,
        "system_prompt": (
            "You are a senior software engineer and technical expert. "
            "Provide precise, structured, and code-focused answers. "
            "When relevant, include corrected code snippets, debugging steps, "
            "and clear technical explanations. Avoid unnecessary fluff."
        )
    },

    "billing": {
        "model": BASE_MODEL,
        "system_prompt": (
            "You are a customer support billing specialist. "
            "Respond in an empathetic, professional tone. "
            "Address payment issues, refunds, subscriptions, and charges clearly. "
            "Explain policies carefully and guide the customer step-by-step."
        )
    },

    "general": {
        "model": BASE_MODEL,
        "system_prompt": (
            "You are a friendly and helpful customer support assistant. "
            "Provide clear and concise responses to general inquiries. "
            "Maintain a polite and approachable tone."
        )
    },

    # --- BONUS: Tool Use Expert ---
    "order_status": {
        "model": BASE_MODEL,
        "system_prompt": (
            "You are an order tracking specialist. "
            "You have access to a tool called get_order_status that returns real-time order information. "
            "Always use this tool when a customer asks about their order. "
            "Present the order details in a clear, friendly format."
        )
    }
}

## Router Function
Classifies each query into one of: `technical`, `billing`, `general`, or `order_status`.

In [4]:
from groq import Groq

client = Groq()

def route_prompt(user_input):
    routing_instruction = f"""
You are an intent classification system.

Classify the following user query into ONE of these categories:
- technical
- billing
- order_status
- general

Use 'order_status' when the user asks about tracking an order, order delivery, shipment, or package.
Return ONLY the category name.
Do not explain.
Do not add extra text.
Do not use punctuation.

User Query:
{user_input}
"""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "user", "content": routing_instruction}
        ],
        temperature=0  # important for consistent classification
    )

    category = response.choices[0].message.content.strip().lower()

    # Safety fallback in case model outputs something unexpected
    if category not in MODEL_CONFIG:
        category = "general"

    return category


## Router Test
Verify the router correctly classifies all four categories.

In [5]:
# Router classification tests
print(route_prompt("My Python script has an IndexError"))       # Expected: technical
print(route_prompt("I was charged twice this month"))           # Expected: billing
print(route_prompt("Hello there!"))                             # Expected: general
print(route_prompt("Where is my order ORD-12345?"))             # Expected: order_status

technical
billing
general
order_status


## Bonus: Tool Use Expert — Order Status

This expert intercepts `order_status` queries and:
1. Defines a tool schema (`get_order_status`) that the LLM can call.
2. The LLM decides to call the tool when it needs order data.
3. We intercept the tool call, run mock data lookup, and feed the result back.
4. The LLM produces a final natural-language response using the tool result.

In [6]:
import json

# ── Mock database ──────────────────────────────────────────────
MOCK_ORDERS = {
    "ORD-12345": {
        "order_id": "ORD-12345",
        "status": "Shipped",
        "estimated_delivery": "2025-04-02",
        "carrier": "FedEx",
        "tracking_number": "FX987654321"
    },
    "ORD-67890": {
        "order_id": "ORD-67890",
        "status": "Processing",
        "estimated_delivery": "2025-04-05",
        "carrier": "UPS",
        "tracking_number": "UPS123456789"
    },
    "ORD-11111": {
        "order_id": "ORD-11111",
        "status": "Delivered",
        "estimated_delivery": "2025-03-28",
        "carrier": "DHL",
        "tracking_number": "DHL000111222"
    }
}

def get_order_status(order_id: str) -> dict:
    """Mock tool: returns order info for a given order ID."""
    order_id = order_id.strip().upper()
    if order_id in MOCK_ORDERS:
        return MOCK_ORDERS[order_id]
    else:
        return {"error": f"Order '{order_id}' not found in our system."}


# ── Tool schema (Groq / OpenAI format) ────────────────────────
ORDER_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": "Retrieves the current status, estimated delivery date, carrier, and tracking number for a given order ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The unique order identifier, e.g. ORD-12345"
                    }
                },
                "required": ["order_id"]
            }
        }
    }
]


def handle_order_status_expert(user_input: str) -> str:
    """
    Tool-use expert for order status queries.
    Step 1 – Ask LLM (with tool schema) what tool to call.
    Step 2 – Execute the mock tool locally.
    Step 3 – Feed tool result back to LLM for final answer.
    """
    system_prompt = MODEL_CONFIG["order_status"]["system_prompt"]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_input}
    ]

    # ── Step 1: First LLM call — model may decide to use the tool ──
    first_response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=messages,
        tools=ORDER_TOOLS,
        tool_choice="auto",
        temperature=0
    )

    response_message = first_response.choices[0].message

    # ── Step 2: Check if the model called the tool ──
    if response_message.tool_calls:
        tool_call = response_message.tool_calls[0]
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)

        print(f"  [Tool Called] {tool_name}({tool_args})")

        # Execute mock tool
        tool_result = get_order_status(tool_args["order_id"])
        print(f"  [Tool Result] {tool_result}")

        # ── Step 3: Feed tool result back to LLM ──
        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_name,
                        "arguments": tool_call.function.arguments
                    }
                }
            ]
        })
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(tool_result)
        })

        final_response = client.chat.completions.create(
            model=BASE_MODEL,
            messages=messages,
            temperature=0.7
        )
        return final_response.choices[0].message.content.strip()

    else:
        # Model answered directly without tool use
        return response_message.content.strip()


## Main Process Request Function
Routes query to the correct expert, including the new Tool Use expert.

In [7]:
def process_request(user_input):
    # Step 1: Classify the query
    category = route_prompt(user_input)
    print(f"  [Routed to: {category}]")

    # Step 2: If order_status, use the Tool Use expert
    if category == "order_status":
        return handle_order_status_expert(user_input)

    # Step 3: Otherwise use the standard expert
    system_prompt = MODEL_CONFIG[category]["system_prompt"]
    response = client.chat.completions.create(
        model=MODEL_CONFIG[category]["model"],
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_input}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content.strip()


## Full Pipeline Tests
Tests all four experts end-to-end.

In [8]:
# ── Technical ────────────────────────────────────────────────
print("=" * 60)
print("TECHNICAL QUERY")
print("=" * 60)
print(process_request("My Python script is throwing an IndexError on line 5."))
print()

# ── Billing ──────────────────────────────────────────────────
print("=" * 60)
print("BILLING QUERY")
print("=" * 60)
print(process_request("I was charged twice for my subscription this month."))
print()

# ── General ──────────────────────────────────────────────────
print("=" * 60)
print("GENERAL QUERY")
print("=" * 60)
print(process_request("Hi! What services do you offer?"))
print()

TECHNICAL QUERY
  [Routed to: technical]
To troubleshoot the issue, we need more information about the script and the error message. Please provide the following:

1. The relevant code snippet (lines 1-10) that's causing the error.
2. The full error message, including the file name, line number, and any other details.
3. The input data or values that are being used when the error occurs.

However, here are some general tips to help you identify and fix the issue:

1. **Check the index**: Ensure that the index you're trying to access is within the bounds of the list or array. Python uses 0-based indexing, so the last valid index is always one less than the length of the collection.
2. **Verify list/array size**: Double-check the size of the list or array by printing its length or using the `len()` function.
3. **Check data types**: Make sure you're not trying to access an element of a different data type, such as a string or a dictionary.

Here's an example of how you might debug the is

In [9]:
# ── BONUS: Order Status with Tool Use ────────────────────────
print("=" * 60)
print("BONUS: ORDER STATUS QUERY (Tool Use Expert)")
print("=" * 60)

print("\n--- Test 1: Known order ---")
print(process_request("Where is my order ORD-12345?"))

print("\n--- Test 2: Another known order ---")
print(process_request("Can you check the status of order ORD-67890?"))

print("\n--- Test 3: Unknown order ---")
print(process_request("What happened to my order ORD-99999?"))

BONUS: ORDER STATUS QUERY (Tool Use Expert)

--- Test 1: Known order ---
  [Routed to: order_status]
  [Tool Called] get_order_status({'order_id': 'ORD-12345'})
  [Tool Result] {'order_id': 'ORD-12345', 'status': 'Shipped', 'estimated_delivery': '2025-04-02', 'carrier': 'FedEx', 'tracking_number': 'FX987654321'}
Your order ORD-12345 has been shipped. You can track its status using the tracking number FX987654321 with FedEx. Estimated delivery is on 2025-04-02.

--- Test 2: Another known order ---
  [Routed to: order_status]
  [Tool Called] get_order_status({'order_id': 'ORD-67890'})
  [Tool Result] {'order_id': 'ORD-67890', 'status': 'Processing', 'estimated_delivery': '2025-04-05', 'carrier': 'UPS', 'tracking_number': 'UPS123456789'}
The status of your order is "Processing". According to our tracking information, your package is expected to be delivered on 2025-04-05. Your order is being shipped via UPS, and the tracking number is UPS123456789. You should be able to track the package 